In [10]:
from unstructured.chunking.title import chunk_by_title
from unstructured.documents.elements import Image, Table
from unstructured_pytesseract import pytesseract
from unstructured.partition.pdf import partition_pdf
from unstructured.partition.text import partition_text as unstructured_partition_text


def partition_txt_file(text_path: str) -> list:
    with open(text_path, "r") as f:
        content = f.read()
    
    elements = unstructured_partition_text(text=content)
    return elements



In [11]:
from unstructured.partition.pdf import partition_pdf as unstructured_partition_pdf

def partition_pdf_file(pdf_path: str) -> list:
    elements = unstructured_partition_pdf(
        filename=pdf_path,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image"],
        extract_image_block_to_payload=True   
    )
    return elements

els = partition_pdf_file("fpga.pdf")

Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

In [12]:
from unstructured.chunking.title import chunk_by_title


def chunk_data(type_file, path):
    if type_file == "text":
        els = partition_txt_file(path)
    elif type_file == "pdf":
        els= partition_pdf_file(path)
    
    chunks = chunk_by_title(
        els,
        max_characters=3000,
        new_after_n_chars=2400,
        combine_text_under_n_chars=500,
        include_orig_elements=True
    )

    # Step 3: Extract from each chunk
    return chunks
    


In [15]:
import base64

def extract_from_chunk(chunk) -> dict:
    result = {
        "text": chunk.text.strip(),
        "images": [],
        "tables": []
    }

    elements_to_check = chunk.metadata.orig_elements or []

    for el in elements_to_check:
        if isinstance(el, Image):
            b64 = getattr(el.metadata, "image_base64", None)
            if b64:
                result["images"].append(b64)

        elif isinstance(el, Table):
            html = getattr(el.metadata, "text_as_html", None)
            result["tables"].append(html if html else el.text)

    return result

In [ ]:
import base64
from openai import OpenAI
from unstructured.documents.elements import Image, Table

client = OpenAI()

def summarize_chunk(chunk_data: dict) -> str:

    content = []

    if chunk_data["text"]:
        content.append({
            "type": "text",
            "text": f"Here is the text content:\n{chunk_data['text']}"
        })
    for b64 in chunk_data["images"]:
        content.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{b64}"
            }
        })

    for table_html in chunk_data["tables"]:
        content.append({
            "type": "text",
            "text": f"Here is a table from the document:\n{table_html}"
        })

    content.append({
        "type": "text",
        "text": "Summarize all the above content concisely. If there are images, describe what they show. If there are tables, summarize the data."
    })

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": content}],
        max_tokens=500
    )
    return response.choices[0].message.content


def get_chunk_output(chunk_data: dict) -> str:
    has_visuals = bool(chunk_data["images"] or chunk_data["tables"])

    if has_visuals:
        return summarize_chunk(chunk_data)
    else:
        return chunk_data["text"]

def embed_chunk(text: str) -> list[float]:
    response = client.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

In [67]:
def file_to_embedds(type: str, path: str) -> list[dict]:
    chunks = chunk_data(type, path)
    results = []

    for i, chunk in enumerate(chunks):
        extracted = extract_from_chunk(chunk)
        content_to_embed = get_chunk_output(extracted)

        results.append({
            "text": extracted["text"],
            "summary": content_to_embed,
            "images": len(extracted["images"]),
            "tables": extracted["tables"],
            "embedding": embed_chunk(content_to_embed),
            "source_file": path,                       
            "file_type": type,                          
            "chunk_index": i,                         
            "page_number": getattr(chunk.metadata, "page_number", None),  
        })

    return results
 
res = file_to_embedds("text", "sample.txt")


In [68]:
import sys
sys.path.append("E:\\MyFiles\\Projects\\Teachers TA AI\\Backend")

from config.database import AsyncSessionLocal, Document
from sqlalchemy import insert

async with AsyncSessionLocal() as session:
    async with session.begin():
        await session.execute(
            insert(Document),
            res  # pass the whole list of dicts
        )

2026-03-23 22:39:19,357 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-23 22:39:19,366 INFO sqlalchemy.engine.Engine INSERT INTO documents (text, summary, images, tables, embedding, source_file, file_type, chunk_index) VALUES ($1::VARCHAR, $2::VARCHAR, $3::INTEGER, $4::JSON, $5, $6::VARCHAR, $7::VARCHAR, $8::INTEGER)
2026-03-23 22:39:19,367 INFO sqlalchemy.engine.Engine [cached since 1631s ago] [('Introduction to Computer Architecture\n\nComputer architecture refers to the set of rules and methods that describe the functionality, organization,  ... (640 characters truncated) ... ges the flow of data between the CPU and other components. Registers are small, fast storage locations within the CPU that hold data being processed.', 'Introduction to Computer Architecture\n\nComputer architecture refers to the set of rules and methods that describe the functionality, organization,  ... (640 characters truncated) ... ges the flow of data between the CPU and other components. Registers

In [54]:
from sqlalchemy import text

async def vector_search(uin):
    query_embedding = embed_chunk(uin)

    async with AsyncSessionLocal() as session:
        result = await session.execute(
            text("""
                SELECT text, summary, source_file
                FROM documents
                ORDER BY embedding <=> :embedding
                LIMIT 3
            """),
            {"embedding": str(query_embedding)}
        )
        rows = result.fetchall()
        print(rows[0][0])
    return rows

In [69]:
async def rag_query(query: str) -> str:
    query_embedding = embed_chunk(query)
    async with AsyncSessionLocal() as session:
        result = await session.execute(
            text("""
                SELECT text FROM documents
                ORDER BY embedding <=> :embedding
                LIMIT 3
            """),
            {"embedding": str(query_embedding)}
        )
        chunks = result.fetchall()
    
    context = "\n\n".join([row[0] for row in chunks])
    
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "Answer the question using only the provided context."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
        ]
    )
    
    return response.choices[0].message.content




In [73]:
await rag_query("when is the next quiz?")

2026-03-23 22:46:51,917 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-23 22:46:51,918 INFO sqlalchemy.engine.Engine 
                SELECT text FROM documents
                ORDER BY embedding <=> $1
                LIMIT 3
            
2026-03-23 22:46:51,919 INFO sqlalchemy.engine.Engine [cached since 551.2s ago] ('[-0.038665771484375, 0.0308837890625, 0.034912109375, -0.022705078125, 0.0020313262939453125, 0.040252685546875, 0.0177764892578125, 0.02850341796875 ... (30444 characters truncated) ... 82373046875, -0.0164794921875, 0.0230712890625, 0.01666259765625, -0.00780487060546875, -0.0537109375, -0.015625, 0.036376953125, -0.006988525390625]',)
2026-03-23 22:46:51,963 INFO sqlalchemy.engine.Engine ROLLBACK


'The next quiz will be on 10th October.'